# LAK-2 — Small files & compaction

**Break → Detect → Fix → Prove.** A streaming/micro-batch writer commits often; every commit
writes at least one data file per partition, so a table with only a few thousand rows ends up
scattered across **hundreds of tiny files**. Counts stay correct, but query planning and scan
I/O degrade. The fix is **compaction** (`rewrite_data_files`).

**Pre-requisite:** the unified Spark server is up (`make up`); this notebook connects via Spark
Connect. **Laptop-safe:** a few thousand rows per write across ~15 appends, all under `.tmp/`;
the **Teardown** cell drops the table (`make clean` clears the rest).

See the companion [`README.md`](./README.md) and the
[troubleshooting sheet](../../docs/troubleshooting.md) (LAK-2 row). This is a **metadata**
pathology — we diagnose it from Iceberg's `.files` table, not the Spark UI Stages tab.

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health, compare_health
from pyspark.sql import functions as F

T = "iceberg_catalog.default.lak2_events"   # Iceberg namespace.table
spark

## Step 0 — Parameters & a fresh table

A tiny `events` table — the kind a streaming job upserts into. We size each batch small on
purpose: the point is **fragmentation**, not volume. Lower `N_APPENDS` if your laptop is slow;
raise it to make the small-files problem more dramatic.

In [ ]:
N_APPENDS    = 15      # number of micro-batch commits (each its own snapshot + files)
ROWS_PER_BATCH = 3_000  # tiny batch — a few thousand events per commit

spark.sql(f"DROP TABLE IF EXISTS {T}")

def batch(start, n, seed):
    # one micro-batch of events (generated lazily, never collected)
    return (spark.range(start, start + n).withColumnRenamed("id", "event_id")
            .withColumn("user_id", F.pmod(F.col("event_id"), F.lit(500)))
            .withColumn("amount", F.round(F.rand(seed=seed) * 100, 2))
            .withColumn("event_type", F.lit("click")))

# create the table from the first batch, then we'll append the rest
batch(0, ROWS_PER_BATCH, seed=0).writeTo(T).using("iceberg").create()
print(f"created {T} with the first batch ({ROWS_PER_BATCH:,} rows)")

## 1. Break it — many small appends

Each `.append()` is a **separate commit**. Iceberg writes at least one data file per partition
per commit (often several at the default write parallelism), so a loop of tiny appends litters
the table with tiny files. This is exactly what a streaming sink does every trigger interval.

In [ ]:
for i in range(1, N_APPENDS):
    batch(i * ROWS_PER_BATCH, ROWS_PER_BATCH, seed=i).writeTo(T).append()

total_rows = spark.table(T).count()
print(f"{N_APPENDS} commits done — {total_rows:,} rows total (tiny!), now scattered across many files")

# Capture the BEFORE health snapshot — the fingerprint of the pathology.
before = table_health(spark, T, "before compaction")
print("data_files :", before["data_files"], "  <- high for so few rows")
print("avg_file   :", before["avg_file_bytes"], "bytes   <- tiny")

## 2. Detect it — read the `.files` metadata

This is a metadata problem, so we read Iceberg's own `<table>.files` metadata table rather than
the Spark UI. Many rows here = many physical files; a tiny average size = fragmentation. (At
query time the same problem shows up as a scan node opening hundreds of files — see
[`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md) on the scan node — but the metadata is
the direct tell.)

In [ ]:
spark.sql(f"""
    SELECT COUNT(*)            AS files,
           ROUND(AVG(file_size_in_bytes), 0) AS avg_bytes,
           SUM(record_count)   AS rows
    FROM {T}.files
""").show()

# a peek at individual files — all tiny:
display_df(spark.sql(f"SELECT file_path, file_size_in_bytes, record_count FROM {T}.files"), limit=10)

## 3. Diagnose

Every commit is atomic and writes **≥1 file per partition** by design — writers never coordinate,
so many small commits mean many small files. Each file carries footer metadata (schema, column
stats) the planner must open and read, and each scan pays a fixed per-file open cost. Hundreds of
tiny files = **planning overhead + poor scan efficiency**, regardless of how few rows they hold.
More CPU won't help — the cost is per-file, not per-row.

## 4. Fix it — bin-pack compaction

Iceberg's `rewrite_data_files` procedure **bin-packs** many small files into fewer large ones in a
single new snapshot. Over Spark Connect we invoke it with `CALL`; the `table` arg is
`namespace.table` (**no** catalog prefix). `min-input-files => 2` lets it compact even our tiny
groups (in production you'd lean on the size-based defaults).

**Prevent it** going forward — set a **target file size** (`write.target-file-size-bytes`, 128 MB
is a common target) so writers aim for fewer, larger files. Combined with fewer/larger writes (the
STR-3 streaming knobs: `maxOffsetsPerTrigger`, longer trigger intervals) and **scheduled
compaction**, this keeps the table from re-fragmenting. The Delta equivalent is `OPTIMIZE <table>`.

In [ ]:
result = spark.sql(f"""
    CALL iceberg_catalog.system.rewrite_data_files(
        table   => 'default.lak2_events',
        options => map('min-input-files','2')
    )
""")
result.show(truncate=False)   # rewritten_data_files_count / added_data_files_count

# prevent re-fragmentation: writers aim for ~128 MB files going forward
spark.sql(f"ALTER TABLE {T} SET TBLPROPERTIES ('write.target-file-size-bytes' = '134217728')")
print("set write.target-file-size-bytes = 128 MB on", T)

## 5. Prove it

Before/after. Watch **Data files** collapse and **Avg file size** rise. Note that **Snapshots**
goes *up* by one (compaction adds a snapshot; it does **not** expire the old ones) — reclaiming
the old files/metadata is LAK-3 (`expire_snapshots`) / LAK-4 (`remove_orphan_files`).

In [ ]:
after = table_health(spark, T, "after compaction")
compare_health([before, after])

## Takeaways & "in real production…"

- **Detect** small files from table **metadata** (`.files` count + avg size), not from row counts —
  a tiny table can still be badly fragmented.
- **Compact on a schedule** (`rewrite_data_files` / Delta `OPTIMIZE`) and **set a target file size**
  (`write.target-file-size-bytes`) so writers don't re-fragment between runs.
- **Prefer fewer, larger writes.** Most small-file problems are born upstream in a streaming job
  writing every trigger — fix the batch sizing first (**STR-3**), compact as a safety net.
- **Compaction is not cleanup.** It rewrites data into new files but leaves old snapshots (and
  their files) in place; pair it with **`expire_snapshots`** (LAK-3) to actually reclaim storage.

## Teardown

We wrote a real table, so we drop it. `make clean` removes everything under `.tmp/`.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
print("Dropped", T, "— (`make clean` clears all of .tmp/ if you want a fresh warehouse).")